# Building Blocks of LangGraph

## Tutorials Covered:
1. Creating Simple Agents
2. Memory in LangGraph
3. Tools Integration
4. Error Handling

## 1. Creating Simple Agents

Learning objectives:
- Build your first LangGraph agent
- Understand agent architecture
- Implement basic agent functionality

In [ ]:
# Tutorial 1: Creating Simple Agents

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

print("=== Creating Simple Agents in LangGraph ===\n")

# Define state for our agent
class AgentState(TypedDict):
    input_query: str
    agent_response: str
    agent_thoughts: str
    conversation_history: list[str]
    needs_clarification: bool

# Define agent nodes

def agent_planner_node(state):
    print(f"Agent planning response for: {state['input_query']}")
    
    # Simple planning logic
    thoughts = f"Analyzing query: {state['input_query']}. Planning appropriate response."
    needs_clarification = "?" in state['input_query'] or len(state['input_query'].split()) < 3
    
    return {
        "agent_thoughts": thoughts,
        "needs_clarification": needs_clarification,
        "conversation_history": state['conversation_history'] + [f"Planned response for: {state['input_query']}"]
    }

def agent_responder_node(state):
    print(f"Agent responding to: {state['input_query']}")
    
    # Generate a simple response based on the input
    response = f"I understand your query about '{state['input_query']}'. This is a simulated response from a LangGraph agent. In a real implementation, this would connect to an LLM for a sophisticated response."
    
    return {
        "agent_response": response,
        "conversation_history": state['conversation_history'] + [f"Responded: {response[:50]}..."]
    }

def agent_clarifier_node(state):
    print(f"Agent requesting clarification for: {state['input_query']}")
    
    clarification_request = f"I need more information to properly answer your query: '{state['input_query']}'. Could you please provide more details?"
    
    return {
        "agent_response": clarification_request,
        "conversation_history": state['conversation_history'] + [f"Requested clarification: {state['input_query']}"]
    }

def agent_summarizer_node(state):
    print(f"Agent summarizing interaction")
    
    summary = f"Interaction summary: Query was '{state['input_query']}', response was '{state['agent_response'][:30]}...', needed clarification: {state['needs_clarification']}"
    
    return {
        "conversation_history": state['conversation_history'] + [summary]
    }

# Routing function to decide next step
def agent_routing_function(state):
    print(f"Agent routing based on needs_clarification: {state['needs_clarification']}")
    
    if state['needs_clarification']:
        return 'clarifier'
    else:
        return 'responder'

print("1. AGENTS in LangGraph are stateful workflows that can plan, act, and respond")
print("   They maintain context and can make decisions based on the state\n")

# Create the agent graph
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("planner", agent_planner_node)
builder.add_node("responder", agent_responder_node)
builder.add_node("clarifier", agent_clarifier_node)
builder.add_node("summarizer", agent_summarizer_node)

# Set up the flow
builder.add_edge("__start__", "planner")
builder.add_conditional_edges(
    "planner",
    agent_routing_function,
    {
        "responder": "responder",
        "clarifier": "clarifier"
    }
)
builder.add_edge("responder", "summarizer")
builder.add_edge("clarifier", "summarizer")
builder.add_edge("summarizer", "__end__")

# Compile the agent
simple_agent = builder.compile()

print("2. Executing the simple agent with different queries:\n")

# Test the agent with different inputs
test_queries = [
    "What is LangGraph?",
    "Explain AI concepts.",
    "How does it work?",  # Short query that might need clarification
    "What are the benefits of using LangGraph for building applications?"
]

for i, query in enumerate(test_queries):
    print(f"--- Test {i+1}: Query '{query}' ---")
    result = simple_agent.invoke({
        "input_query": query,
        "agent_response": "",
        "agent_thoughts": "",
        "conversation_history": [f"Query {i+1} started"],
        "needs_clarification": False
    })
    print(f"Response: {result['agent_response'][:100]}...")
    print(f"Needs clarification: {result['needs_clarification']}")
    print(f"History: {result['conversation_history'][-2:]}")
    print()

print("3. Key concepts of agents in LangGraph:")
print("   - Agents are implemented as stateful graphs")
print("   - They can plan, take actions, and respond")
print("   - Decision-making is possible through conditional edges")
print("   - Context is maintained through the shared state")

## 2. Memory in LangGraph

Learning objectives:
- Manage conversation history and context
- Implement different memory mechanisms
- Understand memory best practices

In [ ]:
# Tutorial 2: Memory in LangGraph

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

print("=== Memory in LangGraph ===\n")

# Define state with memory components
class MemoryState(TypedDict):
    user_input: str
    bot_response: str
    conversation_memory: list[dict]  # Stores conversation history
    short_term_memory: list[str]     # Recent interactions
    long_term_memory: list[str]      # Important information to retain
    memory_summary: str              # Summary of important points

# Node to handle incoming user input and manage memory

def input_handler_node(state):
    print(f"Handling input: {state['user_input']}")
    
    # Add current input to conversation memory
    new_interaction = {
        "role": "user",
        "content": state['user_input'],
        "memory_type": "conversation"
    }
    
    # Update conversation memory
    updated_conversation_memory = state['conversation_memory'] + [new_interaction]
    
    # Update short-term memory (keep only recent 5 items)
    updated_short_term = (state['short_term_memory'] + [f"User: {state['user_input']}"])[-5:]
    
    return {
        "conversation_memory": updated_conversation_memory,
        "short_term_memory": updated_short_term
    }

def context_analyzer_node(state):
    print("Analyzing conversation context from memory")
    
    # Extract relevant context from memory
    recent_context = " ".join(state['short_term_memory'][-3:])  # Last 3 exchanges
    
    # Check if there's anything important to remember for long term
    important_keywords = ["important", "remember", "crucial", "need to keep"]
    contains_important = any(keyword in state['user_input'].lower() for keyword in important_keywords)
    
    # Update long-term memory if needed
    updated_long_term = state['long_term_memory']
    if contains_important:
        updated_long_term = state['long_term_memory'] + [f"Important note from {len(state['conversation_memory'])}th exchange: {state['user_input']}"]
    
    return {
        "long_term_memory": updated_long_term,
        "memory_summary": f"Recent context: {recent_context[:100]}..., Long-term items: {len(updated_long_term)}"
    }

def response_generator_node(state):
    print("Generating response based on memory")
    
    # Create response considering memory
    response = f"I recall we discussed: {'; '.join(state['short_term_memory'][-2:])}. Regarding your current query '{state['user_input']}', I'm taking into account our conversation history. Memory summary: {state['memory_summary']}"
    
    # Add response to conversation memory
    bot_interaction = {
        "role": "bot",
        "content": response,
        "memory_type": "conversation"
    }
    
    return {
        "bot_response": response,
        "conversation_memory": state['conversation_memory'] + [bot_interaction],
        "short_term_memory": state['short_term_memory'] + [f"Bot: {response[:50]}..."]
    }

def memory_cleanup_node(state):
    print("Performing memory cleanup")
    
    # Clean up very old entries from short-term memory
    cleaned_short_term = state['short_term_memory'][-10:]  # Keep only last 10 items
    
    # Create updated summary
    new_summary = f"Current conversation has {len(state['conversation_memory'])} exchanges. Important notes: {len(state['long_term_memory'])}. Recent context: {state['short_term_memory'][-3:] if state['short_term_memory'] else ['None']}"
    
    return {
        "short_term_memory": cleaned_short_term,
        "memory_summary": new_summary
    }

print("1. MEMORY in LangGraph helps maintain context across interactions")
print("   Different types of memory serve different purposes\n")

# Create the memory management graph
builder = StateGraph(MemoryState)

# Add nodes
builder.add_node("input_handler", input_handler_node)
builder.add_node("context_analyzer", context_analyzer_node)
builder.add_node("response_generator", response_generator_node)
builder.add_node("memory_cleanup", memory_cleanup_node)

# Connect nodes in sequence
builder.add_edge("__start__", "input_handler")
builder.add_edge("input_handler", "context_analyzer")
builder.add_edge("context_analyzer", "response_generator")
builder.add_edge("response_generator", "memory_cleanup")
builder.add_edge("memory_cleanup", "__end__")

# Compile the memory graph
memory_graph = builder.compile()

print("2. Executing the memory management graph:\n")

# Simulate a conversation
conversation_turns = [
    "Hello, I want to learn about LangGraph.",
    "Can you explain how state management works?",
    "That's interesting. This is important information to remember.",
    "How does it differ from regular LangChain?",
    "Thanks for the explanation!"
]

# Process each turn maintaining memory
current_state = {
    "user_input": "",
    "bot_response": "",
    "conversation_memory": [],
    "short_term_memory": [],
    "long_term_memory": [],
    "memory_summary": "Starting conversation"
}

for i, user_input in enumerate(conversation_turns):
    print(f"--- Conversation Turn {i+1} ---")
    current_state["user_input"] = user_input
    
    result = memory_graph.invoke(current_state.copy())
    
    # Update current state with result
    current_state.update(result)
    
    print(f"Input: {user_input}")
    print(f"Response: {result['bot_response'][:100]}...")
    print(f"Short-term memory: {len(result['short_term_memory'])} items")
    print(f"Long-term memory: {len(result['long_term_memory'])} items")
    print(f"Memory summary: {result['memory_summary']}")
    print()

print("3. Key concepts of memory in LangGraph:")
print("   - Conversation memory stores the entire interaction history")
print("   - Short-term memory holds recent interactions for immediate context")
print("   - Long-term memory preserves important information")
print("   - Memory management prevents infinite growth and maintains relevance")

## 3. Tools Integration

Learning objectives:
- Connect external tools and APIs to your graph
- Implement tool calling in LangGraph
- Handle tool responses and integration

In [ ]:
# Tutorial 3: Tools Integration

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import json

print("=== Tools Integration in LangGraph ===\n")

# Define state for tools integration
class ToolsState(TypedDict):
    user_query: str
    tool_calls: list[dict]
    tool_responses: list[dict]
    final_response: str
    available_tools: list[str]

# Simulated tools

def calculator_tool(expression: str) -> str:
    """Simulates a calculator tool"""
    try:
        # Safely evaluate the expression (in real applications, use proper expression parsing)
        allowed_chars = set('0123456789+-*/(). ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression"
        
        result = eval(expression)
        return f"Calculation result: {result}"
    except Exception as e:
        return f"Error in calculation: {str(e)}"

def search_tool(query: str) -> str:
    """Simulates a search tool"""
    # In a real implementation, this would call an actual search API
    mock_results = {
        "langgraph": "LangGraph is a library for building stateful, multi-step reasoning applications.",
        "python": "Python is a high-level programming language known for its simplicity and versatility.",
        "ai": "Artificial Intelligence (AI) refers to systems that perform tasks requiring human-like intelligence.",
        "llm": "Large Language Models (LLMs) are AI models trained on vast amounts of text to understand and generate human-like text."
    }
    
    result = mock_results.get(query.lower(), f"No results found for '{query}'. Try a different search term.")
    return f"Search results for '{query}': {result}"

def weather_tool(location: str) -> str:
    """Simulates a weather tool"""
    # Mock weather data
    mock_weather = {
        "new york": "Currently 22°C, sunny with light clouds",
        "london": "Currently 15°C, partly cloudy",
        "tokyo": "Currently 18°C, rainy",
        "paris": "Currently 19°C, overcast"
    }
    
    result = mock_weather.get(location.lower(), f"Weather data not available for {location}")
    return f"Weather in {location}: {result}"

# Define nodes for tool integration

def tool_selector_node(state):
    print(f"Analyzing query to select appropriate tools: {state['user_query']}")
    
    # Determine which tools to use based on the query
    query_lower = state['user_query'].lower()
    selected_tools = []
    
    if any(word in query_lower for word in ["calculate", "compute", "math", "+", "-", "*", "/"]):
        # Extract potential math expression
        import re
        expressions = re.findall(r'[\d+\-*/().]+', state['user_query'])
        if expressions:
            selected_tools.append({
                "tool_name": "calculator",
                "arguments": {"expression": expressions[0]},
                "id": f"calc_{len(state['tool_calls'])}"
            })
    
    if any(word in query_lower for word in ["search", "find", "what is", "who is", "look up"]):
        # Extract search query
        search_query = query_lower.replace("search for", "").replace("find", "").strip()
        if search_query:
            selected_tools.append({
                "tool_name": "search",
                "arguments": {"query": search_query},
                "id": f"search_{len(state['tool_calls'])}"
            })
    
    if any(word in query_lower for word in ["weather", "temperature", "climate"]):
        # Extract location
        location = "New York"  # Default, in real app we'd extract from query
        if "london" in query_lower:
            location = "London"
        elif "tokyo" in query_lower:
            location = "Tokyo"
        elif "paris" in query_lower:
            location = "Paris"
            
        selected_tools.append({
            "tool_name": "weather",
            "arguments": {"location": location},
            "id": f"weather_{len(state['tool_calls'])}"
        })
    
    return {
        "tool_calls": state['tool_calls'] + selected_tools,
        "available_tools": ["calculator", "search", "weather"]
    }

def tool_executor_node(state):
    print(f"Executing {len(state['tool_calls'])} tool calls")
    
    responses = []
    
    for tool_call in state['tool_calls'][len(state['tool_responses']):]:  # Only new calls
        tool_name = tool_call['tool_name']
        arguments = tool_call['arguments']
        tool_id = tool_call['id']
        
        print(f"Executing {tool_name} with args: {arguments}")
        
        if tool_name == "calculator":
            result = calculator_tool(**arguments)
        elif tool_name == "search":
            result = search_tool(**arguments)
        elif tool_name == "weather":
            result = weather_tool(**arguments)
        else:
            result = f"Unknown tool: {tool_name}"
        
        responses.append({
            "tool_id": tool_id,
            "tool_name": tool_name,
            "result": result
        })
    
    return {
        "tool_responses": state['tool_responses'] + responses
    }

def response_synthesizer_node(state):
    print("Synthesizing final response from tool results")
    
    # Combine all tool responses into a coherent answer
    combined_results = "\n".join([
        f"{resp['tool_name'].title()} ({resp['tool_id']}): {resp['result']}" 
        for resp in state['tool_responses']
    ])
    
    final_response = f"Based on your query '{state['user_query']}', here are the results:\n{combined_results}\n\nI hope this answers your question!"
    
    return {
        "final_response": final_response
    }

print("1. TOOLS INTEGRATION allows LangGraph to interact with external functions")
print("   Tools can be APIs, databases, calculators, or any other functions\n")

# Create the tools integration graph
builder = StateGraph(ToolsState)

# Add nodes
builder.add_node("tool_selector", tool_selector_node)
builder.add_node("tool_executor", tool_executor_node)
builder.add_node("response_synthesizer", response_synthesizer_node)

# Connect nodes
builder.add_edge("__start__", "tool_selector")
builder.add_edge("tool_selector", "tool_executor")
builder.add_edge("tool_executor", "response_synthesizer")
builder.add_edge("response_synthesizer", "__end__")

# Compile the tools graph
tools_graph = builder.compile()

print("2. Executing the tools integration graph with different queries:\n")

# Test the tools with different queries
test_queries = [
    "Calculate 25 * 4 + 10",
    "What is LangGraph? Search for information.",
    "What is the weather in London?",
    "Calculate 15 + 30 and also search for Python programming"
]

for i, query in enumerate(test_queries):
    print(f"--- Test {i+1}: Query '{query}' ---")
    result = tools_graph.invoke({
        "user_query": query,
        "tool_calls": [],
        "tool_responses": [],
        "final_response": "",
        "available_tools": []
    })
    print(f"Tool calls made: {len(result['tool_calls'])}")
    print(f"Tool responses: {len(result['tool_responses'])}")
    print(f"Final response: {result['final_response'][:200]}...")
    print()

print("3. Key concepts of tools integration:")
print("   - Tools are external functions that can be called from nodes")
print("   - Tool selection can be automated based on user input")
print("   - Results from tools can be combined to form comprehensive responses")
print("   - Proper error handling is essential when integrating tools")

## 4. Error Handling

Learning objectives:
- Implement proper error management in LangGraph workflows
- Handle exceptions gracefully
- Design robust error recovery mechanisms

In [ ]:
# Tutorial 4: Error Handling

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
import random

print("=== Error Handling in LangGraph ===\n")

# Define state for error handling
class ErrorHandlingState(TypedDict):
    input_data: str
    processing_result: str
    error_occurred: bool
    error_message: str
    retry_count: int
    processing_log: list[str]

# Define nodes for error handling

def input_validator_node(state):
    print(f"Validating input: {state['input_data']}")
    
    # Simulate occasional validation failure
    should_fail = random.random() < 0.2  # 20% chance of failure
    
    if should_fail:
        error_msg = f"Validation failed for input: {state['input_data']}"
        return {
            "error_occurred": True,
            "error_message": error_msg,
            "processing_log": state['processing_log'] + [f"ERROR: {error_msg}"]
        }
    else:
        return {
            "error_occurred": False,
            "processing_log": state['processing_log'] + [f"Validated input: {state['input_data']}"]
        }

def processor_node(state):
    print(f"Processing data: {state['input_data']}")
    
    # Simulate processing that might fail
    should_fail = random.random() < 0.3  # 30% chance of failure
    
    if should_fail:
        error_msg = f"Processing failed for data: {state['input_data']}"
        return {
            "error_occurred": True,
            "error_message": error_msg,
            "processing_log": state['processing_log'] + [f"ERROR: {error_msg}"]
        }
    else:
        result = f"Processed: {state['input_data'].upper()}"
        return {
            "processing_result": result,
            "error_occurred": False,
            "processing_log": state['processing_log'] + [f"Processed successfully: {result}"]
        }

def error_handler_node(state):
    print(f"Handling error: {state['error_message']}")
    
    new_retry_count = state['retry_count'] + 1
    
    # Decide whether to retry based on retry count
    if new_retry_count <= 3:
        return {
            "retry_count": new_retry_count,
            "error_occurred": False,  # Reset error flag to allow retry
            "processing_log": state['processing_log'] + [f"Retry attempt {new_retry_count}"]
        }
    else:
        return {
            "processing_result": f"Failed after {new_retry_count} attempts. Final error: {state['error_message']}",
            "processing_log": state['processing_log'] + [f"Giving up after {new_retry_count} retries"]
        }

def fallback_processor_node(state):
    print(f"Using fallback processing for: {state['input_data']}")
    
    # Fallback processing that rarely fails
    result = f"FALLBACK RESULT for {state['input_data'][:10]}..."
    return {
        "processing_result": result,
        "error_occurred": False,
        "processing_log": state['processing_log'] + [f"Fallback processed: {result}"]
    }

def success_handler_node(state):
    print(f"Handling successful result: {state['processing_result']}")
    return {
        "processing_log": state['processing_log'] + [f"Completed successfully: {state['processing_result']}"]
    }

# Routing function to handle errors
def error_routing_function(state):
    print(f"Routing based on error status: {state['error_occurred']}, retry count: {state['retry_count']}")
    
    if state['error_occurred']:
        if state['retry_count'] < 3:
            return 'error_handler'  # Retry
        else:
            return 'fallback_processor'  # Give up on main processing, use fallback
    else:
        return 'success_handler'

print("1. ERROR HANDLING is crucial for robust LangGraph applications")
print("   Proper error handling ensures graceful degradation and recovery\n")

# Create the error handling graph
builder = StateGraph(ErrorHandlingState)

# Add nodes
builder.add_node("validator", input_validator_node)
builder.add_node("processor", processor_node)
builder.add_node("error_handler", error_handler_node)
builder.add_node("fallback_processor", fallback_processor_node)
builder.add_node("success_handler", success_handler_node)

# Set up the flow
builder.add_edge("__start__", "validator")
builder.add_edge("validator", "processor")

# Add conditional edges from processor based on error status
builder.add_conditional_edges(
    "processor",
    error_routing_function,
    {
        "error_handler": "error_handler",
        "fallback_processor": "fallback_processor",
        "success_handler": "success_handler"
    }
)

# Add conditional edges from error_handler
builder.add_conditional_edges(
    "error_handler",
    lambda state: "processor" if not state['error_occurred'] else "fallback_processor",
    {
        "processor": "processor",
        "fallback_processor": "fallback_processor"
    }
)

# Add final connections
builder.add_edge("fallback_processor", "success_handler")
builder.add_edge("success_handler", "__end__")

# Compile the error handling graph
error_handling_graph = builder.compile()

print("2. Executing the error handling graph with various inputs:\n")

# Test the error handling with multiple runs
test_inputs = ["data1", "data2", "data3", "data4", "data5"]

for i, test_input in enumerate(test_inputs):
    print(f"--- Test Run {i+1}: Input '{test_input}' ---")
    result = error_handling_graph.invoke({
        "input_data": test_input,
        "processing_result": "",
        "error_occurred": False,
        "error_message": "",
        "retry_count": 0,
        "processing_log": [f"Starting processing for {test_input}"]
    })
    print(f"Final result: {result['processing_result']}")
    print(f"Error occurred: {result['error_occurred']}")
    print(f"Log: {result['processing_log'][-3:]}")
    print()

print("3. Key concepts of error handling in LangGraph:")
print("   - Errors should be detected and marked in the state")
print("   - Error handlers can implement retry logic")
print("   - Fallback mechanisms ensure graceful degradation")
print("   - Proper logging helps with debugging and monitoring")